In [1]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [2]:
# loading trained MLP
class SimpleMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))

    def forward(self, x):

        return self.network(x)

In [3]:
model = SimpleMLP()

state_dict = torch.load(
    "mlp_model.pth", map_location=torch.device("cpu"), weights_only=True
)

model.load_state_dict(state_dict)

model.eval()

print("MLP Loaded")

MLP Loaded


In [4]:
# loading scaler
scaler = joblib.load("mlp_scaler.pkl")

print("Scaler Loaded")

Scaler Loaded


In [5]:
# creating dataset to feed to MLP
df = pd.read_csv("natural_probe_dataset.csv")

mlp_df = df[["quality_score", "best_similarity", "margin", "label"]]

mlp_df.to_csv("natural_mlp.csv", index=False)

print("Shape:", mlp_df.shape)

mlp_df.head()

Shape: (294, 4)


,quality_score,best_similarity,margin,label
0,28.981422,0.396520,0.184547,1
1,26.666992,0.546776,0.285556,1
2,36.405415,0.339570,0.101419,1
3,38.936413,0.527681,0.341932,1
4,43.493248,0.451650,0.276032,1


In [6]:
# running MLP
df = pd.read_csv("natural_mlp.csv")

X = df[["quality_score", "best_similarity", "margin"]]

y_true = df["label"]

X_scaled = scaler.transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
with torch.no_grad():

    outputs = model(X_tensor)

    probabilities = torch.softmax(outputs, dim=1)

    y_pred = torch.argmax(outputs, dim=1).numpy()

df["mlp_decision"] = y_pred

df.to_csv("natural_mlp_predictions.csv", index=False)

print("Saved natural_mlp_predictions.csv")

Saved natural_mlp_predictions.csv


In [7]:
# MLP metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred, zero_division=0)

recall = recall_score(y_true, y_pred, zero_division=0)

f1 = f1_score(y_true, y_pred, zero_division=0)

print("===== MLP RESULTS =====\n")

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)
cm_mlp = confusion_matrix(y_true, y_pred)

print("\n===== MLP CONFUSION MATRIX =====")

print(cm_mlp)

print(classification_report(y_true, y_pred, zero_division=0))

===== MLP RESULTS =====

Accuracy : 0.38095238095238093
Precision: 1.0
Recall   : 0.35
F1 Score : 0.5185185185185185

===== MLP CONFUSION MATRIX =====
[[ 14   0]
 [182  98]]
              precision    recall  f1-score   support

           0       0.07      1.00      0.13        14
           1       1.00      0.35      0.52       280

    accuracy                           0.38       294
   macro avg       0.54      0.68      0.33       294
weighted avg       0.96      0.38      0.50       294



In [8]:
# TAR TRR FAR FRR
TN, FP, FN, TP = cm_mlp.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print("\n===== MLP VERIFICATION METRICS =====")

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


===== MLP VERIFICATION METRICS =====
TAR: 0.35
FRR: 0.65
TRR: 1.0
FAR: 0.0


In [9]:
# fixed threshold
threshold = 0.6

threshold_pred = (df["best_similarity"] >= threshold).astype(int)

accuracy_th = accuracy_score(y_true, threshold_pred)

precision_th = precision_score(y_true, threshold_pred, zero_division=0)

recall_th = recall_score(y_true, threshold_pred, zero_division=0)

f1_th = f1_score(y_true, threshold_pred, zero_division=0)

print("===== THRESHOLD RESULTS =====\n")

print("Accuracy :", accuracy_th)

print("Precision:", precision_th)

print("Recall   :", recall_th)

print("F1 Score :", f1_th)
cm_threshold = confusion_matrix(y_true, threshold_pred)

print("\n===== THRESHOLD CONFUSION MATRIX =====")

print(cm_threshold)

print(classification_report(y_true, threshold_pred, zero_division=0))

# TAR FRR FAR TRR
TN, FP, FN, TP = cm_threshold.ravel()

TAR_th = TP / (TP + FN)

FRR_th = FN / (TP + FN)

TRR_th = TN / (TN + FP)

FAR_th = FP / (TN + FP)

print("\n===== THRESHOLD VERIFICATION METRICS =====")

print("TAR:", TAR_th)

print("FRR:", FRR_th)

print("TRR:", TRR_th)

print("FAR:", FAR_th)

===== THRESHOLD RESULTS =====

Accuracy : 0.11904761904761904
Precision: 1.0
Recall   : 0.075
F1 Score : 0.13953488372093023

===== THRESHOLD CONFUSION MATRIX =====
[[ 14   0]
 [259  21]]
              precision    recall  f1-score   support

           0       0.05      1.00      0.10        14
           1       1.00      0.07      0.14       280

    accuracy                           0.12       294
   macro avg       0.53      0.54      0.12       294
weighted avg       0.95      0.12      0.14       294


===== THRESHOLD VERIFICATION METRICS =====
TAR: 0.075
FRR: 0.925
TRR: 1.0
FAR: 0.0


In [11]:
# final comparison table
svm_accuracy = 0.5544217687074829

svm_precision = 1.0
svm_recall = 0.5321428571428571
svm_f1 = 0.6946386946386947

svm_tar = 0.5321428571428571
svm_frr = 0.46785714285714286

svm_trr = 1.0
svm_far = 0.0
comparison = pd.DataFrame(
    {
        "Method": ["Threshold", "SVM", "MLP"],
        "Accuracy": [accuracy_th, svm_accuracy, accuracy],
        "Precision": [precision_th, svm_precision, precision],
        "Recall": [recall_th, svm_recall, recall],
        "F1": [f1_th, svm_f1, f1],
        "TAR": [TAR_th, svm_tar, TAR],
        "FRR": [FRR_th, svm_frr, FRR],
        "TRR": [TRR_th, svm_trr, TRR],
        "FAR": [FAR_th, svm_far, FAR],
    }
)

print("\n===== FINAL COMPARISON =====\n")

comparison.round(4)


===== FINAL COMPARISON =====



,Method,Accuracy,Precision,Recall,F1,TAR,FRR,TRR,FAR
0,Threshold,0.1190,1.0,0.0750,0.1395,0.0750,0.9250,1.0,0.0
1,SVM,0.5544,1.0,0.5321,0.6946,0.5321,0.4679,1.0,0.0
2,MLP,0.3810,1.0,0.3500,0.5185,0.3500,0.6500,1.0,0.0
